# Create Unity Catalog Functions for Policies and Customer Service Tables
Creates reusable SQL functions for querying and analyzing policies and customer service data.

**Source Tables:**
- `llmagent.dev.policies` - Policy information (policy, policy_details, last_updated)
- `llmagent.dev.cust_service_data` - Customer service interactions

**Customer Service Functions:**
1. `get_customer_interactions(email_input)` - Get interactions for a specific email
2. `search_customer_by_email()` - Search customers by email
3. `get_issues_by_category()` - Get issues filtered by category
4. `get_customer_service_summary()` - Get customer service statistics
5. `get_issues_by_date_range()` - Issues within date range
6. `get_agent_performance()` - Agent performance metrics

**Policy Functions:**
7. `get_all_policies()` - Get all policies
8. `search_policy_by_name()` - Search policies by name
9. `search_policy_by_keyword()` - Search policies by keyword
10. `get_recent_policies()` - Get policies updated within N days
11. `get_policies_summary()` - Policy summary statistics

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("CreateUCFunctions").getOrCreate()

# Configuration
CATALOG = "llmagent"
SCHEMA = "dev"
POLICIES_TABLE = f"{CATALOG}.{SCHEMA}.policies"
CUST_SERVICE_TABLE = f"{CATALOG}.{SCHEMA}.cust_service_data"

print(f"Catalog: {CATALOG}")
print(f"Schema: {SCHEMA}")
print(f"Policies Table: {POLICIES_TABLE}")
print(f"Customer Service Table: {CUST_SERVICE_TABLE}")

# CUSTOMER SERVICE FUNCTIONS

## Function 1: Get Customer Interactions by Email

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_customer_interactions(email_input STRING)
    RETURNS TABLE(
        customer_id STRING,
        customer_name STRING,
        email STRING,
        phone STRING,
        interaction_id STRING,
        date_time TIMESTAMP,
        issue_category STRING,
        issue_description STRING,
        agent_id BIGINT
    )
    RETURN
        SELECT
            customer_id,
            name as customer_name,
            email,
            phone_number as phone,
            interaction_id,
            date_time,
            issue_category,
            issue_description,
            agent_id
        FROM {CUST_SERVICE_TABLE}
        WHERE email = email_input
        ORDER BY date_time DESC
""")

print("✓ Function created: get_customer_interactions(email_input)")

## Function 2: Search Customer by Email

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.search_customer_by_email(email_input STRING)
    RETURNS TABLE(
        customer_id STRING,
        customer_name STRING,
        email STRING,
        phone STRING,
        address STRING,
        interaction_count BIGINT,
        last_interaction TIMESTAMP
    )
    RETURN
        SELECT
            customer_id,
            name as customer_name,
            email,
            phone_number as phone,
            address,
            COUNT(interaction_id) as interaction_count,
            MAX(date_time) as last_interaction
        FROM {CUST_SERVICE_TABLE}
        WHERE email = email_input
        GROUP BY customer_id, name, email, phone_number, address
""")

print("✓ Function created: search_customer_by_email(email_input)")

## Function 3: Get Issues by Category

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_issues_by_category(category_input STRING)
    RETURNS TABLE(
        customer_id STRING,
        customer_name STRING,
        interaction_id STRING,
        date_time TIMESTAMP,
        issue_category STRING,
        issue_description STRING,
        agent_id BIGINT
    )
    RETURN
        SELECT
            customer_id,
            name as customer_name,
            interaction_id,
            date_time,
            issue_category,
            issue_description,
            agent_id
        FROM {CUST_SERVICE_TABLE}
        WHERE issue_category = category_input
        ORDER BY date_time DESC
""")

print("✓ Function created: get_issues_by_category(category_input)")

## Function 4: Get Customer Service Summary

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_customer_service_summary()
    RETURNS TABLE(
        total_customers BIGINT,
        total_interactions BIGINT,
        unique_issues BIGINT,
        active_agents BIGINT,
        avg_interactions_per_customer DOUBLE,
        earliest_interaction TIMESTAMP,
        latest_interaction TIMESTAMP
    )
    RETURN
        SELECT
            COUNT(DISTINCT customer_id) as total_customers,
            COUNT(interaction_id) as total_interactions,
            COUNT(DISTINCT issue_category) as unique_issues,
            COUNT(DISTINCT agent_id) as active_agents,
            ROUND(COUNT(interaction_id) / COUNT(DISTINCT customer_id), 2) as avg_interactions_per_customer,
            MIN(date_time) as earliest_interaction,
            MAX(date_time) as latest_interaction
        FROM {CUST_SERVICE_TABLE}
""")

print("✓ Function created: get_customer_service_summary()")

## Function 5: Get Issues by Date Range

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_issues_by_date_range(start_date TIMESTAMP, end_date TIMESTAMP)
    RETURNS TABLE(
        customer_id STRING,
        customer_name STRING,
        interaction_id STRING,
        date_time TIMESTAMP,
        issue_category STRING,
        issue_description STRING,
        agent_id BIGINT
    )
    RETURN
        SELECT
            customer_id,
            name as customer_name,
            interaction_id,
            date_time,
            issue_category,
            issue_description,
            agent_id
        FROM {CUST_SERVICE_TABLE}
        WHERE date_time BETWEEN start_date AND end_date
        ORDER BY date_time DESC
""")

print("✓ Function created: get_issues_by_date_range(start_date, end_date)")

## Function 6: Get Agent Performance

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_agent_performance()
    RETURNS TABLE(
        agent_id BIGINT,
        total_interactions BIGINT,
        unique_customers BIGINT,
        unique_issue_categories BIGINT,
        earliest_interaction TIMESTAMP,
        latest_interaction TIMESTAMP
    )
    RETURN
        SELECT
            agent_id,
            COUNT(interaction_id) as total_interactions,
            COUNT(DISTINCT customer_id) as unique_customers,
            COUNT(DISTINCT issue_category) as unique_issue_categories,
            MIN(date_time) as earliest_interaction,
            MAX(date_time) as latest_interaction
        FROM {CUST_SERVICE_TABLE}
        GROUP BY agent_id
        ORDER BY total_interactions DESC
""")

print("✓ Function created: get_agent_performance()")

# POLICY FUNCTIONS

## Function 7: Get All Policies

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_all_policies()
    RETURNS TABLE(
        policy STRING,
        policy_details STRING,
        last_updated DATE
    )
    RETURN
        SELECT
            policy,
            policy_details,
            last_updated
        FROM {POLICIES_TABLE}
        ORDER BY last_updated DESC
""")

print("✓ Function created: get_all_policies()")

## Function 8: Search Policy by Name

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.search_policy_by_name(policy_name_input STRING)
    RETURNS TABLE(
        policy STRING,
        policy_details STRING,
        last_updated DATE
    )
    RETURN
        SELECT
            policy,
            policy_details,
            last_updated
        FROM {POLICIES_TABLE}
        WHERE policy LIKE CONCAT('%', policy_name_input, '%')
        ORDER BY last_updated DESC
""")

print("✓ Function created: search_policy_by_name(policy_name_input)")

## Function 9: Search Policy by Keyword

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.search_policy_by_keyword(keyword_input STRING)
    RETURNS TABLE(
        policy STRING,
        policy_details STRING,
        last_updated DATE
    )
    RETURN
        SELECT
            policy,
            policy_details,
            last_updated
        FROM {POLICIES_TABLE}
        WHERE policy_details LIKE CONCAT('%', keyword_input, '%')
           OR policy LIKE CONCAT('%', keyword_input, '%')
        ORDER BY last_updated DESC
""")

print("✓ Function created: search_policy_by_keyword(keyword_input)")

## Function 10: Get Recent Policies

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_recent_policies(days_input INT)
    RETURNS TABLE(
        policy STRING,
        policy_details STRING,
        last_updated DATE,
        days_since_update INT
    )
    RETURN
        SELECT
            policy,
            policy_details,
            last_updated,
            DATEDIFF(CURRENT_DATE(), last_updated) as days_since_update
        FROM {POLICIES_TABLE}
        WHERE DATEDIFF(CURRENT_DATE(), last_updated) <= days_input
        ORDER BY last_updated DESC
""")

print("✓ Function created: get_recent_policies(days_input)")

## Function 11: Get Policies Summary

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_policies_summary()
    RETURNS TABLE(
        total_policies BIGINT,
        oldest_policy_date DATE,
        newest_policy_date DATE,
        avg_policy_length INT
    )
    RETURN
        SELECT
            COUNT(*) as total_policies,
            MIN(last_updated) as oldest_policy_date,
            MAX(last_updated) as newest_policy_date,
            ROUND(AVG(LENGTH(policy_details))) as avg_policy_length
        FROM {POLICIES_TABLE}
""")

print("✓ Function created: get_policies_summary()")

## List All Created Functions

In [ ]:
display(spark.sql(f"""
    SHOW FUNCTIONS IN {CATALOG}.{SCHEMA} LIKE 'get_*'
"""))

## Test Functions

In [ ]:
# Test: Get customer service summary
print("Test 1: Customer Service Summary")
display(spark.sql(f"""
    SELECT * FROM {CATALOG}.{SCHEMA}.get_customer_service_summary()
"""))

In [ ]:
# Test: Get policies summary
print("Test 2: Policies Summary")
display(spark.sql(f"""
    SELECT * FROM {CATALOG}.{SCHEMA}.get_policies_summary()
"""))

In [ ]:
# Test: Get agent performance
print("Test 3: Agent Performance Statistics")
display(spark.sql(f"""
    SELECT * FROM {CATALOG}.{SCHEMA}.get_agent_performance() LIMIT 5
"""))

In [ ]:
# Test: Get all policies
print("Test 4: All Policies")
display(spark.sql(f"""
    SELECT * FROM {CATALOG}.{SCHEMA}.get_all_policies() LIMIT 5
"""))

## Summary

### Customer Service Functions:
1. `get_customer_interactions(email_input)` - Get interactions for specific customer email
2. `search_customer_by_email(email_input)` - Search customer and get interaction count
3. `get_issues_by_category(category_input)` - Get all issues in a category
4. `get_customer_service_summary()` - Overall customer service statistics
5. `get_issues_by_date_range(start_date, end_date)` - Get issues within date range
6. `get_agent_performance()` - Agent performance metrics

### Policy Functions:
7. `get_all_policies()` - Return all policies
8. `search_policy_by_name(policy_name_input)` - Search policy by name
9. `search_policy_by_keyword(keyword_input)` - Search policy by keyword
10. `get_recent_policies(days_input)` - Get policies updated within N days
11. `get_policies_summary()` - Policy statistics

### Usage Examples:
```sql
-- Customer interactions by email
SELECT * FROM llmagent.dev.get_customer_interactions('customer@email.com');

-- All policies
SELECT * FROM llmagent.dev.get_all_policies();

-- Search policies by keyword
SELECT * FROM llmagent.dev.search_policy_by_keyword('coverage');

-- Recent policies (last 30 days)
SELECT * FROM llmagent.dev.get_recent_policies(30);

-- Policy summary
SELECT * FROM llmagent.dev.get_policies_summary();
```